# 04 · Build and validate reporting tables

**Question:** How can the cleaned sales and model results be presented consistently in Power BI?

**Inputs:** `data/processed/cereal_analysis.parquet` from notebook 01, three model CSVs from notebook 03, and `sql/01_reporting_views.sql`.
**Outputs:** a local `data/pricing.duckdb` database and five CSVs in `outputs/powerbi/`.

The SQL reporting views are read from the tracked SQL file, which is the single source of their definitions. This notebook no longer duplicates and overwrites that file. Re-running replaces the local tables and exports.

## 1. Connect and import the analysis records

Rename fields for reporting, retain UPC as text, and create a binary reporting flag for nonempty promotion codes. `sales` retains the source-record grain; it is not deduplicated. Preview a few rows to confirm the reporting schema.

In [ ]:
from pathlib import Path

# Kernels may start in the repository root or the notebooks directory.
PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
if not (PROJECT / "notebooks").is_dir() or not (PROJECT / "README.md").is_file():
    raise RuntimeError("Start the notebook from the repository root or notebooks directory.")

import duckdb

PYTHON_RESULTS = PROJECT / "outputs" / "python_results"
POWERBI = PROJECT / "outputs" / "powerbi"
POWERBI.mkdir(parents=True, exist_ok=True)
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(PROJECT / "data" / "pricing.duckdb"))


sales_path = (
    PROJECT / "data" / "processed" / "cereal_analysis.parquet"
).as_posix()

con.execute(f"""
CREATE OR REPLACE TABLE sales AS
SELECT
    CAST(STORE AS INTEGER) AS store_id,
    CAST(UPC AS VARCHAR) AS upc,
    CAST(WEEK AS INTEGER) AS week,
    CAST(MOVE AS BIGINT) AS units,
    CAST(UNIT_PRICE AS DOUBLE) AS unit_price,
    CAST(REVENUE AS DOUBLE) AS revenue,
    TRIM(DESCRIP) AS product,
    TRIM(SIZE) AS size,
    CASE
        WHEN SALE IS NULL OR TRIM(SALE) = '' THEN 0
        ELSE 1
    END AS promo_recorded
FROM read_parquet('{sales_path}');
""")

con.sql("SELECT * FROM sales LIMIT 5").df()

## 2. Import the model results

Read the three exports from notebook 03. Explicit text typing protects the UPC join key from numeric inference. Keep the model identifiers and scenario assumptions; the portfolio and case study use different promotion controls.

In [ ]:
for table_name in [
    "product_elasticities",
    "pricing_scenarios",
    "pricing_grid"
]:
    file_path = (
        PYTHON_RESULTS / f"{table_name}.csv"
    ).as_posix()

    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT *
        FROM read_csv(
            '{file_path}',
            header = true,
            types = {{'upc': 'VARCHAR'}}
        );
    """)

con.sql("SHOW TABLES").df()

## 3. Audit record grain and product keys

Record counts, coverage, and totals establish a baseline before aggregation. Repeated store–product–week keys are diagnostic, not automatically duplicates to remove.

The product-metadata query should return no rows: conflicting labels for the same UPC could create multiple dimension rows and inflate joined totals. Resolve source metadata if conflicts appear.

In [ ]:
print(con.sql("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT upc) AS products,
    COUNT(DISTINCT store_id) AS stores,
    MIN(week) AS first_week,
    MAX(week) AS last_week,
    SUM(units) AS total_units,
    SUM(revenue) AS total_revenue
FROM sales;
""").df().to_string(index=False))

print(con.sql("""
SELECT
    store_id,
    upc,
    week,
    COUNT(*) AS row_count
FROM sales
GROUP BY store_id, upc, week
HAVING COUNT(*) > 1
LIMIT 20;
""").df().to_string(index=False))

metadata_conflicts = con.sql("""
SELECT
    upc
FROM sales
GROUP BY upc
HAVING
    COUNT(DISTINCT product) > 1
    OR COUNT(DISTINCT size) > 1;
""").df()
if not metadata_conflicts.empty:
    raise ValueError("Conflicting product descriptions or sizes; inspect metadata_conflicts before reporting.")

## 4. Create the reporting views

`dim_product` provides the product label used in reports. `weekly_sales` aggregates all stores to **UPC × week × recorded-promotion flag**, summing units and revenue and counting source records. Store-level detail is intentionally absent from that export.

Keeping the definitions in one SQL file makes changes reviewable and prevents notebook and SQL versions from drifting.

In [ ]:
reporting_sql = (PROJECT / "sql" / "01_reporting_views.sql").read_text()
con.execute(reporting_sql)

duplicate_products = con.sql("""
    SELECT upc FROM dim_product GROUP BY upc HAVING COUNT(*) > 1
""").df()
if not duplicate_products.empty:
    raise ValueError("dim_product must contain exactly one row per UPC.")

con.sql("SELECT * FROM weekly_sales LIMIT 10").df()

## 5. Reconcile totals before exporting

Aggregation must preserve total units and revenue. Both differences should be zero (revenue is compared to cents). Stop export if totals do not reconcile. This catches unintended filtering or aggregation changes before they reach a dashboard.

In [ ]:
reconciliation = con.sql("""
SELECT
    (SELECT SUM(units) FROM sales)
        - (SELECT SUM(units) FROM weekly_sales)
        AS units_difference,

    ROUND(
        (SELECT SUM(revenue) FROM sales)
        - (SELECT SUM(revenue) FROM weekly_sales),
        2
    ) AS revenue_difference;
""").df()
if not (reconciliation.iloc[0] == 0).all():
    raise ValueError("Reporting totals do not reconcile; inspect reconciliation before exporting.")
reconciliation

## 6. Export the five Power BI tables

The two sales-reporting views and three model tables are written as CSVs without a dataframe index. These are the files to use for dashboard refreshes; CSV itself does not preserve types, so keep UPC typed as text when importing to Power BI. The database connection is closed after export.

In [ ]:
tables = [
    "dim_product",
    "weekly_sales",
    "product_elasticities",
    "pricing_scenarios",
    "pricing_grid"
]

for table_name in tables:
    result = con.sql(f"SELECT * FROM {table_name}").df()
    destination = POWERBI / f"{table_name}.csv"
    result.to_csv(destination, index=False)

    print(f"{table_name}: {len(result):,} rows exported")

con.close()